# Streaming

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Access the AI_MODEL environment variable
ai_model = os.getenv("AI_MODEL")

In [2]:
from langchain.chat_models import init_chat_model

# initialize a commercial chat model (e.g., "openai:gpt-5") or use a local
# model (e.g., "ollama:gpt-oss")
model = init_chat_model(ai_model, temperature=0)

In [3]:
from langchain.agents import create_agent

# create your agent! Add the model object you just created, a prompt etc.
agent = create_agent(
    model=model,
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [4]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Why do programmers prefer dark mode? Because light attracts bugs.

Want another?


## values
You have seen this streaming mode in our examples so far. 

In [5]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Why did the scarecrow win an award? Because he was outstanding in his field.

Want another groan-worthy one?


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [6]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

Sunrise tiptoes in on a spoon of light,  
Pancakes giggle as we flip them just right.  
The cat conducts a breakfast-band parade,  
While Grandpa’s hat joins in, though off-key and swayed.

We build a blanket fort—no adults allowed!—  
A secret kingdom where imagination’s loud.  
Socks take vacations in the laundry’s deep sea;  
They return as mismatched sailors, proud as can be.

We splash through puddles that mirror the sky,  
Then chase paper boats until the wind says goodbye.  
Cookies are traded for stories at night,  
Each tale a lantern, each laugh a bright light.

Stars tuck us in with a silvery grin,  
Every day’s a new page where our adventures begin.  
Hand in hand, through silly and true,  
Home is wherever we’re together—me and you.

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [7]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='f3db3299-2b02-493a-96f6-94d1efc60b8a')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='f3db3299-2b02-493a-96f6-94d1efc60b8a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 132, 'total_tokens': 284, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQEuKJbfE7zXI56adoNodQSA6irCT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logpr

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])